# 14 - Weak-label training, expert-label evaluation

Frozen ImageNet ResNet-18 slice features (as in notebook 09), mean-pooled per plane,
with a linear 12-target head trained on **all 4,407 studies**:

* targets: expert labels where available, otherwise the report-derived weak score;
* loss: **masked** binary cross-entropy; a target the report never mentions has weight 0,
  expert-labelled rows have weight `GOLD_WEIGHT`;
* features are standardised with statistics from the training folds only;
* the epoch count is picked on a held-out slice of the *weakly labelled* training
  studies, so the 58 expert labels are never used for any choice;
* evaluation: pooled out-of-fold AUC on the 58 expert-labelled studies, per target and
  macro (targets without both classes are excluded and counted).

There is no augmentation: the encoder is frozen and features are computed once.
Augmentation (medprep `MedicalPipeline(training=True)`) belongs to the fine-tuning
milestone.

**Settings:** GPU, internet ON (downloads torchvision ResNet-18 weights once; they are
saved with the outputs so notebook 15 runs offline).
**Inputs:** notebook 11 (`weak_labels_v1.csv`), 12 (`folds_v1.csv`), 13 (cache + manifest).
**Outputs:** `features_v1.npz`, `model/` (encoder, fold heads, config), `oof_gold_v1.csv`,
`metrics_v1.json`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# The competition mount path has differed between Kaggle runtimes; take the first that has train.csv.
_SLUG = "rsna-knee-abnormality-detection"
_roots = [Path("/kaggle/input/competitions") / _SLUG, Path("/kaggle/input") / _SLUG,
          *sorted(Path("/kaggle/input").glob("*/" + _SLUG)), *sorted(Path("/kaggle/input").glob("*"))]
ROOT = next((r for r in _roots if (r / "train.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("competition train.csv not found; /kaggle/input has: "
                            + str(sorted(str(p) for p in Path("/kaggle/input").glob("*/*"))[:50]))
print("competition data:", ROOT)
OUT = Path("/kaggle/working")
TARGET_COLUMNS = [
    "ACL", "MCL",
    "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]
SEED = 42
from pathlib import Path


def find_input(name):
    """Locate one attached upstream file by name.

    Only notebook outputs and datasets are searched: walking the competition
    mount would visit millions of DICOM files.
    """
    hits = []
    for base in (Path("/kaggle/input/notebooks"), Path("/kaggle/input/datasets"),
                 Path("/kaggle/input")):
        if base.is_dir():
            for child in base.iterdir():
                if (child.is_dir()
                        and child.name not in ("competitions", "notebooks", "datasets")
                        and not (child / "train_series").exists()
                        and not (child / "test_series").exists()):
                    hits += list(child.rglob(name))
    hits = sorted(set(hits))
    if len(hits) != 1:
        raise FileNotFoundError(
            f"expected exactly one attached '{name}', found {len(hits)}: {hits[:5]}. "
            "Attach the upstream notebook output via Add Input."
        )
    return hits[0]
# Shared feature extractor: identical in notebooks 14 and 15.
import torch
from torch import nn
from torchvision.models import resnet18

FEATURE_VERSION = "resnet18-imagenet-frozen-slice-mean-3planes+presence"
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def build_encoder(state_dict=None):
    from torchvision.models import ResNet18_Weights
    weights = None if state_dict is not None else ResNet18_Weights.IMAGENET1K_V1
    enc = resnet18(weights=weights)
    enc.fc = nn.Identity()
    if state_dict is not None:
        enc.load_state_dict(state_dict)
    return enc.eval().requires_grad_(False)


@torch.inference_mode()
def study_features(encoder, volumes, present, device):
    """volumes: (B, 3, 16, 224, 224) uint8; present: (B, 3).
    Returns (B, 3*512 + 3): per-plane slice-mean features, zeros for a missing
    plane, followed by the presence mask."""
    b, p, d, h, w = volumes.shape
    x = volumes.to(device).reshape(b * p * d, 1, h, w).float().div_(255)
    x = (x.repeat(1, 3, 1, 1) - IMAGENET_MEAN.to(device)) / IMAGENET_STD.to(device)
    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
        f = encoder(x)
    f = f.float().reshape(b, p, d, -1).mean(dim=2) * present.to(device)[..., None]
    return torch.cat([f.reshape(b, -1), present.to(device)], dim=1).cpu()

import json
import random

GOLD_WEIGHT = 3.0
EPOCHS = 400
LR = 1e-2
WEIGHT_DECAY = 1e-3
INNER_FRACTION = 0.1


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 1. Inputs

In [ ]:
weak = pd.read_csv(find_input("weak_labels_v1.csv"), dtype={"StudyInstanceUID": str})
folds = pd.read_csv(find_input("folds_v1.csv"), dtype={"StudyInstanceUID": str})
manifest_path = find_input("preprocess_manifest_v1.csv")
manifest = pd.read_csv(manifest_path, dtype={"StudyInstanceUID": str}).drop_duplicates("StudyInstanceUID", keep="last")
CACHE = manifest_path.parent / "cache"
assert manifest["version"].eq(manifest["version"].iloc[0]).all()
PREPROC_VERSION = manifest["version"].iloc[0]

data = weak.merge(folds[["StudyInstanceUID", "fold", "group"]], on="StudyInstanceUID",
                  validate="one_to_one")
data = data.merge(manifest[["StudyInstanceUID", "status"]], on="StudyInstanceUID", how="left")
missing = data["status"].ne("ok")
print("studies without a usable cache:", int(missing.sum()),
      "(of which expert-labelled:", int((missing & data["is_gold"].eq(1)).sum()), ")")
data = data[~missing].reset_index(drop=True)

## 2. Features (computed once, cached)

In [ ]:
from torch.utils.data import DataLoader, Dataset


class CacheDataset(Dataset):
    def __init__(self, studies):
        self.studies = list(studies)

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, i):
        with np.load(CACHE / f"{self.studies[i]}.npz") as z:
            return torch.from_numpy(z["volume"]), torch.from_numpy(z["present"])


encoder = build_encoder().to(device)
loader = DataLoader(CacheDataset(data["StudyInstanceUID"]), batch_size=8, num_workers=4)
from tqdm.auto import tqdm
X = torch.cat([study_features(encoder, v, p, device)
               for v, p in tqdm(loader, desc="features", mininterval=30)]).numpy()
assert X.shape == (len(data), 3 * 512 + 3) and np.isfinite(X).all()
np.savez_compressed(OUT / "features_v1.npz", X=X, studies=data["StudyInstanceUID"].to_numpy())
print("features:", X.shape)

## 3. Targets and weights

In [ ]:
gold_mask = data["is_gold"].eq(1).to_numpy()
Y = np.zeros((len(data), 12), np.float32)
W = np.zeros((len(data), 12), np.float32)
for j, t in enumerate(TARGET_COLUMNS):
    Y[:, j] = np.where(gold_mask, data[t + "__gold"].fillna(0), data[t + "__weak"])
    W[:, j] = np.where(gold_mask, GOLD_WEIGHT, 1.0 - data[t + "__silent"])
print("effective weak targets per column:", (W[~gold_mask] > 0).sum(axis=0).tolist())

## 4. Train one head per fold; evaluate out-of-fold on expert labels

In [ ]:
def fit_head(Xtr, Ytr, Wtr, Xin, Yin, Win, seed):
    """Full-batch AdamW on masked BCE; keeps the epoch with the lowest inner-weak loss."""
    torch.manual_seed(seed)
    xt, yt, wt = (torch.from_numpy(a).to(device) for a in (Xtr, Ytr, Wtr))
    xi, yi, wi = (torch.from_numpy(a).to(device) for a in (Xin, Yin, Win))
    head = nn.Linear(Xtr.shape[1], 12).to(device)
    prior = ((Ytr * Wtr).sum(0) / np.maximum(Wtr.sum(0), 1e-6)).clip(1e-3, 1 - 1e-3)
    with torch.no_grad():
        head.weight.zero_()
        head.bias.copy_(torch.from_numpy(np.log(prior / (1 - prior))).float())
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    bce = nn.functional.binary_cross_entropy_with_logits

    def loss(x, y, w):
        return (bce(head(x), y, reduction="none") * w).sum() / w.sum().clamp_min(1e-6)

    best = (float("inf"), 0, {k: v.clone() for k, v in head.state_dict().items()})
    for epoch in range(1, EPOCHS + 1):
        head.train()
        opt.zero_grad()
        loss(xt, yt, wt).backward()
        opt.step()
        head.eval()
        with torch.no_grad():
            inner = loss(xi, yi, wi).item()
        if inner < best[0]:
            best = (inner, epoch, {k: v.clone() for k, v in head.state_dict().items()})
    head.load_state_dict(best[2])
    return head.eval(), best[1]


MODEL = OUT / "model"
MODEL.mkdir(exist_ok=True)
oof = np.full((len(data), 12), np.nan, np.float32)
fold_info = []
for k in sorted(data["fold"].unique()):
    train_idx = np.flatnonzero(data["fold"].ne(k))
    # Inner split for epoch choice: weakly labelled studies only, whole groups.
    rng = np.random.default_rng(SEED + int(k))
    weak_groups = data.loc[train_idx[~gold_mask[train_idx]], "group"].unique()
    inner_groups = set(rng.choice(weak_groups, int(INNER_FRACTION * len(weak_groups)), replace=False))
    is_inner = data.loc[train_idx, "group"].isin(inner_groups).to_numpy() & ~gold_mask[train_idx]
    fit_idx, inner_idx = train_idx[~is_inner], train_idx[is_inner]

    mean = X[fit_idx].mean(0, keepdims=True)
    std = X[fit_idx].std(0, keepdims=True)
    std = np.where(std < 1e-6, 1.0, std)
    z = lambda a: ((a - mean) / std).astype(np.float32)

    head, epoch = fit_head(z(X[fit_idx]), Y[fit_idx], W[fit_idx],
                           z(X[inner_idx]), Y[inner_idx], W[inner_idx], SEED + int(k))
    eval_idx = np.flatnonzero(data["fold"].eq(k))
    with torch.no_grad():
        oof[eval_idx] = torch.sigmoid(head(torch.from_numpy(z(X[eval_idx])).to(device))).cpu().numpy()
    torch.save({"state_dict": head.state_dict(), "mean": mean, "std": std, "epoch": epoch},
               MODEL / f"head_fold{k}.pt")
    fold_info.append({"fold": int(k), "fit": len(fit_idx), "inner": len(inner_idx),
                      "eval": len(eval_idx), "eval_gold": int(gold_mask[eval_idx].sum()),
                      "epoch": epoch})
display(pd.DataFrame(fold_info))

## 5. Metrics on the 58 expert-labelled studies

In [ ]:
from sklearn.metrics import roc_auc_score


def safe_auc(y, p):
    """AUC, or NaN when only one class is present (the metric is undefined)."""
    y = np.asarray(y)
    return float(roc_auc_score(y, p)) if np.unique(y).size == 2 else np.nan


def auc_table(gold, scores, n_boot=2000, seed=0):
    """Per-target AUC with a percentile bootstrap interval over studies."""
    rng = np.random.default_rng(seed)
    rows = []
    for t in TARGET_COLUMNS:
        y, p = gold[t].to_numpy(), scores[t].to_numpy()
        boots = []
        for _ in range(n_boot):
            i = rng.integers(0, len(y), len(y))
            a = safe_auc(y[i], p[i])
            if np.isfinite(a):
                boots.append(a)
        rows.append({"target": t, "auc": safe_auc(y, p),
                     "ci_low": np.percentile(boots, 2.5) if boots else np.nan,
                     "ci_high": np.percentile(boots, 97.5) if boots else np.nan,
                     "n_pos": int(y.sum()), "n_neg": int((1 - y).sum())})
    return pd.DataFrame(rows)


def macro_auc(table):
    """Challenge metric: mean of per-target AUCs; undefined targets excluded and counted."""
    return float(table["auc"].mean(skipna=True)), int(table["auc"].notna().sum())

g = data[gold_mask]
gold_y = g[[t + "__gold" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1).astype(int)
pred = pd.DataFrame(oof[gold_mask], columns=TARGET_COLUMNS, index=g.index)
regex = g[[t + "__weak" for t in TARGET_COLUMNS]].set_axis(TARGET_COLUMNS, axis=1)

image_t = auc_table(gold_y, pred)
image_t["regex_report_auc"] = auc_table(gold_y, regex)["auc"]
display(image_t.round(3))
macro, n_eval = macro_auc(image_t)

per_fold = []
for k, part in g.groupby("fold"):
    t = pd.DataFrame([{"auc": safe_auc(gold_y.loc[part.index, c], pred.loc[part.index, c])}
                      for c in TARGET_COLUMNS])
    per_fold.append({"fold": int(k), "macro_auc": t["auc"].mean(), "evaluable": int(t["auc"].notna().sum())})
display(pd.DataFrame(per_fold).round(3))
print(f"pooled OOF macro AUC (image model): {macro:.4f} over {n_eval}/12 targets")
print(f"reference: report extractor {macro_auc(image_t.assign(auc=image_t['regex_report_auc']))[0]:.4f} "
      "(not available at test time); notebook 10 gold-only image baseline 0.526; constant 0.5")

## 6. Save model bundle, config and predictions

In [ ]:
import hashlib


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


torch.save(encoder.state_dict(), MODEL / "encoder_resnet18.pt")
prevalence = {t: float(gold_y[t].mean()) for t in TARGET_COLUMNS}
config = {
    "preproc_version": PREPROC_VERSION, "feature_version": FEATURE_VERSION,
    "target_columns": TARGET_COLUMNS, "seed": SEED, "gold_weight": GOLD_WEIGHT,
    "epochs_max": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "inner_fraction": INNER_FRACTION, "folds": fold_info, "fallback_prevalence": prevalence,
    "inputs_sha256": {n: sha(find_input(n)) for n in ("weak_labels_v1.csv", "folds_v1.csv")},
    "versions": {"torch": torch.__version__, "numpy": np.__version__},
}
(MODEL / "config.json").write_text(json.dumps(config, indent=2))
pd.concat([g[["StudyInstanceUID", "fold"]], pred.add_suffix("__pred")], axis=1).to_csv(
    OUT / "oof_gold_v1.csv", index=False)
(OUT / "metrics_v1.json").write_text(json.dumps({
    "oof_macro_auc": macro, "evaluable_targets": n_eval,
    "per_target": image_t.to_dict("records"), "per_fold": per_fold}, indent=2, default=float))
print("saved model bundle to", MODEL)